# 05b — Head OS/platform (thử nghiệm)

Thêm một tầng **trên** ba head cũ: `os` ∈ {iOS, Android, Windows, macOS, Linux, Embedded}.
Thứ tự trả lời **OS → type → make → model**, mỗi tầng tự abstain.

Lý do: ba head cũ chỉ biết 12 thiết bị FIELD, gặp thiết bị lạ là `__unknown__` hết. Nhưng
**OS lộ ra qua vân tay cấu trúc** (PRL, VCI, dịch vụ mDNS) và vân tay đó dùng chung cho
mọi máy cùng OS — `realme-6i` chưa từng train vẫn gửi đúng kiểu DHCP của Android.

| Tầng | Bằng chứng | Ghi chú |
|---|---|---|
| L0 | hostname mặc định (`iPhone`, `DESKTOP-…`, `realme-…`), VCI `MSFT 5.0`, VCI `android-dhcp-*` **+ MAC ngẫu nhiên** | chuỗi thiết bị tự khai |
| L1 | tập option DHCP gần nhất (Jaccard ≥ 0.9), dịch vụ Apple Continuity qua mDNS **+ MAC ngẫu nhiên** | tri thức giao thức, chịu được extractor sort lại PRL |
| L2 | rừng trên feature cấu trúc — **không** `dns_tokens`/`tls_sni_tokens` | chỉ được nói OS đa dụng khi cửa sổ có DHCP hoặc mDNS |

`Embedded` = firmware thiết bị chuyên dụng (camera, loa, ổ cắm, TV…) **kể cả khi nhân là
Linux/Android**: câu hỏi sản phẩm là "có phải máy tính đa dụng không". Echo Studio gửi
`android-dhcp-7.1.2` nhưng là loa — tách nó khỏi điện thoại bằng MAC: 5/5 điện thoại FIELD
ngẫu nhiên hoá MAC, 0/39 thiết bị CIC thì không.

**Giới hạn đã biết:** chưa có mẫu macOS nào (lớp chỉ trả lời được qua hostname); iOS chỉ 1
thiết bị train (L1-mDNS gánh phần tổng quát hoá); Linux 2 thiết bị.

In [1]:
# Mô tả: Cấu hình biến môi trường và số luồng cho BLAS/TF
import os

os.environ['OPENBLAS_NUM_THREADS'] = '44'  # 50% cores
os.environ['MKL_NUM_THREADS'] = '44'
os.environ['OMP_NUM_THREADS'] = '44'
os.environ['NUMEXPR_NUM_THREADS'] = '44'

# TensorFlow threading
os.environ['TF_NUM_INTRAOP_THREADS'] = '44'  # Parallel ops
os.environ['TF_NUM_INTEROP_THREADS'] = '8'   # Independent ops

# turn off oneDNN optimization if needed
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

print("Configured for 88-core CPU")

Configured for 88-core CPU


In [2]:
# Nạp định nghĩa dùng chung từ 00_common.ipynb (không train).
from pathlib import Path

_cwd = Path.cwd().resolve()
_PIPELINE = next((d for p in (_cwd, *_cwd.parents)
                  for d in (p, p / "pipeline", p / "Code" / "pipeline")
                  if (d / "00_common.ipynb").is_file()), None)
assert _PIPELINE is not None, f"Không thấy pipeline/00_common.ipynb quanh {_cwd}"

if not globals().get("SDC_DEFS_LOADED"):
    SDC_IMPORT_ONLY = True
    try:
        get_ipython().run_line_magic("run", f'-i "{_PIPELINE / "00_common.ipynb"}"')
    finally:
        del SDC_IMPORT_ONLY

import json
from datetime import datetime
from collections import Counter, OrderedDict

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
import re
import onnx
import onnxruntime as ort
from sklearn.ensemble import RandomForestClassifier

ort.set_default_logger_severity(3)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 60)

Configured for 88-core CPU
Gốc dự án: D:\01.AI_Security\02.SDC


Đã nạp hàm SDC từ 00_common.ipynb

## 1. Nhãn OS

FIELD gán theo bảng dưới; mọi thiết bị CIC-2022 là `Embedded`. Thiết bị FIELD mới mà chưa
có trong bảng thì **dừng hẳn** — đừng để nó lặng lẽ thành Embedded.

In [3]:
UNK = "__unknown__"
OS_CLASSES = ["iOS", "Android", "Windows", "macOS", "Linux", "Embedded"]
OS_THRESHOLD = 0.6          # ngưỡng của rừng (L2); L0/L1 là luật, conf = 1.0

FIELD_OS = {
    "iPhone": "iOS",
    "OPPO A92": "Android", "Samsung Phone (Duc Anh)": "Android",
    "Xiaomi Redmi Note 10": "Android", "Xiaomi Redmi Note 14 Pro": "Android",
    "Desktop PC DNGJHRT": "Windows", "Desktop PC DucAnh": "Windows",
    "Laptop VAF70SQ6": "Windows", "Lee Kingdom Laptop": "Windows",
    "Linova Laptop (linova)": "Linux", "Raspberry Pi": "Linux",
    "IP Camera (field)": "Embedded",
}


def mac_is_random(mac):
    """Bit U/L (0x02) của octet đầu. MAC hỏng -> 0."""
    m = str(mac).strip().lower().replace("-", ":")
    if not re.fullmatch(r"[0-9a-f]{2}(:[0-9a-f]{2}){5}", m):
        return 0
    return int(int(m[:2], 16) & 0x02 != 0)


sessions, _ = load_sessions(SESSIONS_PATH)
sessions = sessions.copy()
is_field = sessions["scenario"].eq("FIELD")
unmapped = sorted(set(sessions.loc[is_field, "canonical_device"]) - set(FIELD_OS))
assert not unmapped, f"thiết bị FIELD chưa có nhãn OS: {unmapped}"
sessions["os"] = np.where(is_field, sessions["canonical_device"].map(FIELD_OS), "Embedded")
sessions["mac_is_random"] = sessions["mac"].map(mac_is_random)

summary = sessions.groupby("os").agg(thiết_bị=("canonical_device", "nunique"), cửa_sổ=("mac", "size"),
                                     mac_ngẫu_nhiên=("mac_is_random", "mean"))
display(summary.round(2))

,thiết_bị,cửa_sổ,mac_ngẫu_nhiên
os,,,
Android,4,10,1.0
Embedded,40,8145,0.0
Linux,2,9,0.0
Windows,4,22,0.0
iOS,1,3,1.0


## 2. Feature cấu trúc và cách cân mẫu

Bỏ `dns_tokens` và `tls_sni_tokens`: đó là *nội dung* (thiết bị gọi về đâu), đổi theo người
dùng — chính là thứ khiến head cũ học thuộc lòng. Thêm `mac_is_random`.

Cân **theo thiết bị**: mỗi thiết bị nặng như nhau rồi cân lớp — 331 cửa sổ của một ổ cắm
không được đè lên 3 cửa sổ của iPhone.

In [4]:
TFIDF_MAX_FEATURES = {col: None for col in TFIDF_MAX_FEATURES}   # như 05
STRUCT_CAT = ["dhcp_prl", "tls_fp", "tls_version", "tls_alpn", "tls_ciphers"]
STRUCT_TEXT = ["dhcp_vci", "mdns_tokens"]


def os_feature_cols(df):
    num = [c for c in df.columns if c.startswith(("dhcp_opt_", "has_"))] + ["dhcp_prl_len"]
    return num + ["mac_is_random"] + STRUCT_CAT + STRUCT_TEXT


def device_weights(df, label="os"):
    w = 1.0 / df.groupby("canonical_device")["canonical_device"].transform("size")
    per_class = w.groupby(df[label]).transform("sum")
    return (w / per_class).to_numpy() * len(df) / df[label].nunique()


def prl_set(prl):
    text = str(prl)
    return frozenset() if text in NOT_PRESENT else frozenset(int(t) for t in re.findall(r"\d+", text))


def mine_signatures(df):
    """Tập option DHCP -> tập OS đã thấy. TẬP chứ không chuỗi: extractor của đối tác sort
    lại thứ tự PRL (`1:108:114:...`) — so tập thì vẫn khớp."""
    sig = {}
    rows = df.loc[~df["dhcp_prl"].astype(str).isin(NOT_PRESENT), ["dhcp_prl", "os"]].drop_duplicates()
    for prl, os_ in rows.itertuples(index=False):
        sig.setdefault(prl_set(prl), set()).add(os_)
    return sig


def fit_os(df, cols):
    enc = fit_encoder(df, cols)
    X, _, _ = apply_encoder(df, enc)
    forest = RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=0)
    forest.fit(X, df["os"].to_numpy(), sample_weight=device_weights(df))
    return {"encoder": enc, "model": forest, "cols": cols, "signatures": mine_signatures(df)}


NOT_PRESENT = ("", MISSING, "nan", "None")
OS_COLS = os_feature_cols(sessions)
print(f"{len(OS_COLS)} cột cấu trúc")

39 cột cấu trúc

## 3. Quyết định OS: L0 → L1 → L2

In [5]:
HOST_OS = [
    (r"^(iphone|ipad|ipod)\b", "iOS"),
    (r"^(macbook|imac|mac-?mini|mac-?pro|mac-?studio)\b", "macOS"),
    (r"^(android|galaxy|sm-|redmi|xiaomi|poco|mi-|realme|oppo|vivo|oneplus|pixel|huawei|honor|nokia|moto)", "Android"),
    (r"^(desktop|laptop|win)-[a-z0-9]{4,}$", "Windows"),
    (r"^(raspberrypi|ubuntu|debian|fedora|archlinux)\b", "Linux"),
    (r"^(esp[_-]|tuya|shelly|tasmota|wled)", "Embedded"),
]


def l0_os(hostname, vci, mac_random):
    """Chuỗi thiết bị tự khai."""
    h = str(hostname or "").strip().lower()
    if h and h not in ("<missing>", "nan", "none"):
        for pattern, os_ in HOST_OS:
            if re.search(pattern, h):
                return os_, "hostname"
    v = str(vci or "").strip().lower()
    if v.startswith("msft "):
        return "Windows", "vci"
    # android-dhcp-* cũng có ở loa Echo Studio (MAC thật) -> chỉ tin khi MAC ngẫu nhiên.
    if v.startswith("android-dhcp-") and mac_random:
        return "Android", "vci"
    return None, None


def l1_os(signatures, prl, mdns, mac_random, min_jaccard=0.9):
    """Vân tay giao thức: tập option DHCP gần nhất, rồi Apple Continuity qua mDNS."""
    s = prl_set(prl)
    if s:
        best, owners = 0.0, set()
        for key, value in signatures.items():
            j = len(s & key) / len(s | key)
            if j > best + 1e-9:
                best, owners = j, set(value)
            elif abs(j - best) <= 1e-9:
                owners |= value
        if best >= min_jaccard and len(owners) == 1:
            os_ = next(iter(owners))
            # Vân tay thiết bị gia dụng mà MAC ngẫu nhiên hoá: bằng chứng trái nhau, nhường L2.
            if not (os_ == "Embedded" and mac_random):
                return os_, f"prl~{best:.2f}"
    # Handoff/AirDrop: Apple TV/HomePod cũng quảng bá nhưng dùng MAC thật.
    m = str(mdns)
    if mac_random and ("_companion link" in m or "_rdlink" in m):
        return "iOS", "mdns"
    return None, None


def os_decide(bundle, frame, thr=OS_THRESHOLD, use_hostname=True):
    """-> list (nhãn, nguồn, conf) cho từng dòng."""
    X, _, _ = apply_encoder(frame[bundle["cols"]], bundle["encoder"])
    proba = bundle["model"].predict_proba(X)
    classes = bundle["model"].classes_
    out = []
    for i, (_, r) in enumerate(frame.iterrows()):
        rnd = int(r.get("mac_is_random", 0))
        label, src = l0_os(r.get("dhcp_hostname") if use_hostname else None, r.get("dhcp_vci"), rnd)
        if not label:
            label, src = l1_os(bundle["signatures"], r.get("dhcp_prl"), r.get("mdns_tokens"), rnd)
        if label:
            out.append((label, src, 1.0))
            continue
        k = int(proba[i].argmax()); p, conf = classes[k], float(proba[i, k])
        # Vân tay TLS là của TRÌNH DUYỆT (Chrome trên Windows = Chrome trên Linux), thứ tự
        # cipher của Apple xáo theo phiên: cửa sổ chỉ-TLS không đủ để nói một OS đa dụng.
        tls_only = not (r.get("has_dhcp") == 1 or r.get("has_mdns") == 1)
        if conf >= thr and not (p != "Embedded" and tls_only):
            out.append((p, "forest", conf))
        else:
            out.append((UNK, "abstain", conf))
    return out

## 4. Cascade OS → type → make → model

* type/make **mâu thuẫn** OS -> abstain tầng đó (vd. OS=Windows mà type=Smartphone).
* tầng dưới **abstain** mà OS đủ nói thay -> điền: iOS → Smartphone + Apple;
  Android **+ MAC ngẫu nhiên** → Smartphone; Windows/macOS → Laptop; macOS → Apple.
* make hoặc type abstain thì model cũng abstain.

In [6]:
TYPE_OK = {"iOS": {"Smartphone"}, "Android": {"Smartphone"}, "Windows": {"Laptop"},
           "macOS": {"Laptop"}, "Linux": {"Laptop", "Single-board Computer"}}
GENERAL_PURPOSE_TYPES = {"Smartphone", "Laptop", "Single-board Computer"}
MAKE_OK = {"iOS": {"Apple"}, "macOS": {"Apple"}}


def cascade(os_label, mac_random, make, type_, model):
    """-> (os, make, type, model, ghi chú)."""
    notes = []
    if os_label != UNK:
        allowed = TYPE_OK.get(os_label)
        if os_label == "Embedded" and type_ in GENERAL_PURPOSE_TYPES:
            notes.append(f"type {type_} mâu thuẫn Embedded"); type_ = UNK
        elif type_ != UNK and allowed and type_ not in allowed:
            notes.append(f"type {type_} mâu thuẫn {os_label}"); type_ = UNK
        if type_ == UNK:
            if os_label == "iOS" or (os_label == "Android" and mac_random):
                type_ = "Smartphone"; notes.append("type suy từ OS")
            elif os_label in ("Windows", "macOS"):
                type_ = "Laptop"; notes.append("type suy từ OS")
        allowed = MAKE_OK.get(os_label)
        if make != UNK and allowed and make not in allowed:
            notes.append(f"make {make} mâu thuẫn {os_label}"); make = UNK
        if make == "Apple" and os_label not in ("iOS", "macOS"):
            notes.append(f"make Apple mâu thuẫn {os_label}"); make = UNK
        if make == UNK and allowed:
            make = "Apple"; notes.append("make suy từ OS")
    if (make == UNK or type_ == UNK) and model != UNK:
        notes.append("model bỏ vì make/type abstain"); model = UNK
    return os_label, make, type_, model, "; ".join(notes)

## 5. Đánh giá trung thực: giữ lại theo **thiết bị**

OS phải tổng quát hoá sang thiết bị chưa thấy, nên mỗi thiết bị FIELD là một fold, thiết bị
CIC chia 6 nhóm. Encoder, rừng **và bảng chữ ký L1** đều dựng lại trên phần train của fold.
Không dùng hostname — đo phần học được, không đo phần tự khai.

In [7]:
cic_devices = sorted(sessions.loc[~is_field, "canonical_device"].unique())
rng = np.random.default_rng(0)
group = {d: f"cic{k}" for d, k in zip(cic_devices, rng.permutation(np.arange(len(cic_devices)) % 6))}
group.update({d: d for d in sessions.loc[is_field, "canonical_device"].unique()})
fold = sessions["canonical_device"].map(group).to_numpy()

oof_label = np.empty(len(sessions), dtype=object)
oof_src = np.empty(len(sessions), dtype=object)
for g in np.unique(fold):
    train, test = fold != g, fold == g
    b = fit_os(sessions[train], OS_COLS)
    for i, (label, src, _) in zip(np.where(test)[0], os_decide(b, sessions[test], use_hostname=False)):
        oof_label[i], oof_src[i] = label, src.split("~")[0]
sessions["os_pred"], sessions["os_src"] = oof_label, oof_src

rows = []
for os_ in OS_CLASSES:
    m = sessions["os"].eq(os_)
    if not m.any():
        continue
    ok = (m & sessions.os_pred.eq(os_)).sum()
    bad = (m & sessions.os_pred.ne(os_) & sessions.os_pred.ne(UNK)).sum()
    rows.append({"os": os_, "thiết bị": sessions.loc[m, "canonical_device"].nunique(),
                 "cửa sổ": int(m.sum()), "đúng": int(ok), "sai": int(bad), "abstain": int(m.sum() - ok - bad)})
display(pd.DataFrame(rows).set_index("os"))
print("nguồn quyết định:", sessions.os_src.value_counts().to_dict())


def vote(g):
    answered = g[g.os_pred.ne(UNK)]
    return answered.os_pred.mode().iat[0] if len(answered) else UNK


by_device = sessions.groupby("canonical_device").apply(
    lambda g: pd.Series({"os": g.os.iat[0], "đoán": vote(g), "cửa sổ": len(g),
                         "đã trả lời": int(g.os_pred.ne(UNK).sum())}), include_groups=False)
dev_ok = int(by_device.os.eq(by_device["đoán"]).sum())
dev_bad = int((by_device["đoán"].ne(UNK) & by_device.os.ne(by_device["đoán"])).sum())
print(f"\nMức thiết bị (bỏ phiếu các cửa sổ đã trả lời): đúng {dev_ok}/{len(by_device)}, sai {dev_bad}")
display(by_device[by_device.os.ne("Embedded") | by_device["đoán"].ne("Embedded")])

,thiết bị,cửa sổ,đúng,sai,abstain
os,,,,,
iOS,1,3,3,0,0
Android,4,10,4,0,6
Windows,4,22,5,0,17
Linux,2,9,4,0,5
Embedded,40,8145,7648,8,489


nguồn quyết định:

{'forest': 6878, 'prl': 782, 'abstain': 517, 'vci': 9, 'mdns': 3}


Mức thiết bị (bỏ phiếu các cửa sổ đã trả lời): đúng 48/51, sai 0

,os,đoán,cửa sổ,đã trả lời
canonical_device,,,,
Desktop PC DNGJHRT,Windows,Windows,5,1
Desktop PC DucAnh,Windows,Windows,6,3
LG TV,Embedded,__unknown__,3,0
Laptop VAF70SQ6,Windows,__unknown__,6,0
Lee Kingdom Laptop,Windows,Windows,5,1
Linova Laptop (linova),Linux,Linux,6,2
OPPO A92,Android,Android,1,1
Raspberry Pi,Linux,Linux,3,2
Samsung Phone (Duc Anh),Android,Android,6,2


## 6. Train bản cuối và lưu

In [8]:
os_bundle = fit_os(sessions, OS_COLS)
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_os")
run_dir = MODELS / RUN_ID
run_dir.mkdir(parents=True)
joblib.dump({"format": "sdc-os-v0", "classes": OS_CLASSES, "threshold": OS_THRESHOLD,
             "field_os": FIELD_OS, **os_bundle}, run_dir / "model.joblib", compress=3)
(run_dir / "meta.json").write_text(json.dumps({
    "format": "sdc-os-v0", "run_id": RUN_ID, "created": datetime.now().isoformat(timespec="seconds"),
    "dataset": str(SESSIONS_PATH), "n_sessions": int(len(sessions)),
    "classes": OS_CLASSES, "threshold": OS_THRESHOLD, "feature_cols": OS_COLS,
    "n_signatures": len(os_bundle["signatures"]),
    "oof_device": {"đúng": dev_ok, "sai": dev_bad, "tổng": int(len(by_device))},
    "note": "Thử nghiệm. Chưa có mẫu macOS; iOS 1 thiết bị, Linux 2. Chưa vào ONNX.",
}, indent=2, ensure_ascii=False), encoding="utf-8")
print("Đã lưu:", run_dir)

Đã lưu:

D:\01.AI_Security\02.SDC\Models\20261002_163153_os

## 7. Thử trên thiết bị chưa thấy, ghép với model deploy

Ba head cũ lấy từ ONNX `*_iden` mới nhất (12 cột); OS lấy từ bundle vừa train.

In [9]:
IDEN = sorted(MODELS.glob("*_iden/sdc_iden.onnx"))[-1]
iden = ort.InferenceSession(str(IDEN), providers=["CPUExecutionProvider"])
IDEN_COLS = json.loads({p.key: p.value for p in onnx.load(str(IDEN), load_external_data=False)
                        .metadata_props}["sdc.contract"])["columns"]
print("model deploy:", IDEN.parent.name, f"({len(IDEN_COLS)} cột)")


def mac_oui(mac):
    m = str(mac).strip().lower().replace("-", ":")
    ok = re.fullmatch(r"[0-9a-f]{2}(:[0-9a-f]{2}){5}", m)
    return m[:8] if ok and not mac_is_random(m) else MISSING


def prep(frame):
    f = frame.copy()
    f["mac_is_random"] = f["mac"].map(mac_is_random)
    f["mac_oui"] = f["mac"].map(mac_oui)
    for c in ("dhcp_hostname", "mdns_model"):
        f[c] = f[c].fillna(MISSING).astype(str) if c in f.columns else MISSING
    for c in set(OS_COLS) | set(IDEN_COLS):
        numeric = c.startswith(("dhcp_opt_", "has_")) or c == "dhcp_prl_len"
        if c not in f.columns:
            f[c] = 0 if numeric else MISSING
        else:  # dòng thiếu nguồn (vd. mới có hostname) -> ô NaN
            f[c] = f[c].fillna(0).astype(int) if numeric else f[c].fillna(MISSING)
    return f


def iden_matrix(f):
    out = f[IDEN_COLS].copy()
    for c in IDEN_COLS:
        v = out[c]
        out[c] = v.astype(np.float32).astype(str) if pd.api.types.is_numeric_dtype(v) else v.astype(str)
    return out.to_numpy(dtype=object)


def explain(frame, use_hostname=True):
    f = prep(frame)
    if not use_hostname:
        f["dhcp_hostname"] = MISSING
    old = iden.run(["output"], {"input": iden_matrix(f)})[0]
    osd = os_decide(os_bundle, f)
    rows = []
    for i, (_, r) in enumerate(f.iterrows()):
        label, src, conf = osd[i]
        o, make, type_, model, note = cascade(label, r["mac_is_random"], *old[i])
        rows.append({"mac": r["mac"], "hostname": r["dhcp_hostname"],
                     "trước (make/type/model)": "/".join(old[i]),
                     "os": f"{o} [{src}]", "sau (make/type/model)": f"{make}/{type_}/{model}",
                     "ghi chú": note})
    return pd.DataFrame(rows)

model deploy:

20261001_170502_iden

(12 cột)

In [10]:
# 7a. pcap tự thu: 3 MAC không có trong bảng nhãn (2 hostname 'iphone' + 1 MAC Fingerbank nói Apple)
feat = DATA / "features"
cap = pd.read_csv(feat / "session_features_capture.csv")
dhcp = pd.read_csv(feat / "dhcp_features_capture.csv")
cap["dhcp_hostname"] = cap["mac"].map(dhcp.dropna(subset=["hostname"]).drop_duplicates("mac")
                                      .set_index("mac")["hostname"])
labelled_macs = set(sessions.loc[is_field, "mac"])
unseen = cap[~cap["mac"].isin(labelled_macs)]
for use_hostname in (True, False):
    print(f"--- {unseen['mac'].nunique()} MAC chưa thấy, {'có' if use_hostname else 'BỎ'} hostname")
    display(explain(unseen, use_hostname))

--- 3 MAC chưa thấy, có hostname

,mac,hostname,trước (make/type/model),os,sau (make/type/model),ghi chú
0,16:76:ae:9b:f1:a0,iphone,Apple/Smartphone/iPhone,iOS [hostname],Apple/Smartphone/iPhone,
1,16:76:ae:9b:f1:a0,iphone,Apple/Smartphone/iPhone,iOS [hostname],Apple/Smartphone/iPhone,
2,06:27:04:ee:2a:40,iphone,Apple/Smartphone/iPhone,iOS [hostname],Apple/Smartphone/iPhone,
3,16:76:ae:9b:f1:a0,iphone,Apple/Smartphone/iPhone,iOS [hostname],Apple/Smartphone/iPhone,
4,06:27:04:ee:2a:40,iphone,Apple/Smartphone/iPhone,iOS [hostname],Apple/Smartphone/iPhone,
5,06:27:04:ee:2a:40,iphone,Apple/Smartphone/iPhone,iOS [hostname],Apple/Smartphone/iPhone,
6,16:76:ae:9b:f1:a0,iphone,Apple/Smartphone/iPhone,iOS [hostname],Apple/Smartphone/iPhone,
7,6e:4c:77:fc:06:d3,<missing>,__unknown__/Smartphone/__unknown__,iOS [prl~1.00],Apple/Smartphone/__unknown__,make suy từ OS
8,6e:4c:77:fc:06:d3,<missing>,__unknown__/__unknown__/__unknown__,iOS [mdns],Apple/Smartphone/__unknown__,type suy từ OS; make suy từ OS
9,16:76:ae:9b:f1:a0,iphone,Apple/Smartphone/iPhone,iOS [hostname],Apple/Smartphone/iPhone,


--- 3 MAC chưa thấy, BỎ hostname

,mac,hostname,trước (make/type/model),os,sau (make/type/model),ghi chú
0,16:76:ae:9b:f1:a0,<missing>,__unknown__/__unknown__/__unknown__,__unknown__ [abstain],__unknown__/__unknown__/__unknown__,
1,16:76:ae:9b:f1:a0,<missing>,Apple/Smartphone/iPhone,iOS [mdns],Apple/Smartphone/iPhone,
2,06:27:04:ee:2a:40,<missing>,__unknown__/Smartphone/__unknown__,iOS [mdns],Apple/Smartphone/__unknown__,make suy từ OS
3,16:76:ae:9b:f1:a0,<missing>,__unknown__/__unknown__/__unknown__,__unknown__ [abstain],__unknown__/__unknown__/__unknown__,
4,06:27:04:ee:2a:40,<missing>,__unknown__/__unknown__/__unknown__,iOS [mdns],Apple/Smartphone/__unknown__,type suy từ OS; make suy từ OS
5,06:27:04:ee:2a:40,<missing>,__unknown__/Smartphone/__unknown__,iOS [prl~1.00],Apple/Smartphone/__unknown__,make suy từ OS
6,16:76:ae:9b:f1:a0,<missing>,__unknown__/Smartphone/__unknown__,iOS [prl~0.90],Apple/Smartphone/__unknown__,make suy từ OS
7,6e:4c:77:fc:06:d3,<missing>,__unknown__/Smartphone/__unknown__,iOS [prl~1.00],Apple/Smartphone/__unknown__,make suy từ OS
8,6e:4c:77:fc:06:d3,<missing>,__unknown__/__unknown__/__unknown__,iOS [mdns],Apple/Smartphone/__unknown__,type suy từ OS; make suy từ OS
9,16:76:ae:9b:f1:a0,<missing>,__unknown__/__unknown__/__unknown__,__unknown__ [abstain],__unknown__/__unknown__/__unknown__,


In [11]:
# 7b. test_pcap_collection: thiết bị đã biết, MAC mới, ngày khác
test3 = pd.read_csv(feat / "test_pcap_collection" / "session_features.csv")
test3["dhcp_hostname"] = test3["device"]
for use_hostname in (False, True):
    print(f"--- test_pcap_collection, {'có' if use_hostname else 'BỎ'} hostname")
    display(explain(test3, use_hostname))

--- test_pcap_collection, BỎ hostname

,mac,hostname,trước (make/type/model),os,sau (make/type/model),ghi chú
0,e2:1c:d2:b0:af:a5,<missing>,__unknown__/Smartphone/__unknown__,Android [vci],__unknown__/Smartphone/__unknown__,
1,7e:f9:e6:c0:a1:aa,<missing>,__unknown__/Smartphone/__unknown__,Android [vci],__unknown__/Smartphone/__unknown__,
2,6e:18:8d:88:08:15,<missing>,__unknown__/Smartphone/__unknown__,iOS [prl~1.00],Apple/Smartphone/__unknown__,make suy từ OS


--- test_pcap_collection, có hostname

,mac,hostname,trước (make/type/model),os,sau (make/type/model),ghi chú
0,e2:1c:d2:b0:af:a5,Redmi-Note-10,Xiaomi/Smartphone/Redmi Note 10,Android [hostname],Xiaomi/Smartphone/Redmi Note 10,
1,7e:f9:e6:c0:a1:aa,Samsung-cua-Duc-Anh,Samsung/Smartphone/__unknown__,Android [vci],Samsung/Smartphone/__unknown__,
2,6e:18:8d:88:08:15,iPhone,Apple/Smartphone/iPhone,iOS [hostname],Apple/Smartphone/iPhone,


In [12]:
# 7c. iPhone trong log đối tác: PRL bị sort, có option 162 chưa từng thấy, không TLS.
# 7d. realme-6i trong log router: mới có MAC + hostname; thêm một bản GIẢ LẬP gói DHCP Android.
fc = [c for c in sessions.columns if c.startswith(("dhcp_opt_", "has_"))] + ["dhcp_prl_len"] + \
     STRUCT_CAT + ["dhcp_vci", "dns_tokens", "mdns_tokens", "tls_sni_tokens"]
partner = aggregate([{"proto": "dhcp", "opt55": "1,108,114,119,121,15,162,252,3,6"}]
                    + [{"proto": "dns", "qname": q} for q in ("captive.apple.com", "gateway.icloud.com")]
                    + [{"proto": "mdns", "qname": q} for q in ("_companion-link._tcp.local", "_rdlink")], fc)
realme_sim = aggregate([{"proto": "dhcp", "opt55": "1,3,6,15,26,28,51,58,59,43,114,108",
                         "opt60": "android-dhcp-11"}], fc)
demo = pd.DataFrame([
    dict(partner, mac="fa:4c:89:0d:da:5f", dhcp_hostname=None),
    {"mac": "22:5e:21:5d:d4:53", "dhcp_hostname": "realme-6i"},
    dict(realme_sim, mac="22:5e:21:5d:d4:53", dhcp_hostname=None),
], index=["iPhone đối tác, không hostname", "realme-6i, chỉ hostname", "realme-6i GIẢ LẬP, không hostname"])
display(explain(demo).set_index(demo.index))

,mac,hostname,trước (make/type/model),os,sau (make/type/model),ghi chú
"iPhone đối tác, không hostname",fa:4c:89:0d:da:5f,<missing>,__unknown__/__unknown__/__unknown__,iOS [prl~0.90],Apple/Smartphone/__unknown__,type suy từ OS; make suy từ OS
"realme-6i, chỉ hostname",22:5e:21:5d:d4:53,realme-6i,__unknown__/__unknown__/__unknown__,Android [hostname],__unknown__/Smartphone/__unknown__,type suy từ OS
"realme-6i GIẢ LẬP, không hostname",22:5e:21:5d:d4:53,<missing>,__unknown__/__unknown__/__unknown__,Android [vci],__unknown__/Smartphone/__unknown__,type suy từ OS


In [13]:
# 7e. Hồi quy: cascade không được làm hỏng câu đúng, không được biến IoT thành điện thoại/PC.
labelled = prep(cap[cap["mac"].isin(labelled_macs)].merge(
    sessions.loc[is_field, ["mac", "make", "type", "model"]].drop_duplicates("mac"), on="mac"))
old = iden.run(["output"], {"input": iden_matrix(labelled)})[0]
osd = os_decide(os_bundle, labelled)
broken = 0
for i, (_, r) in enumerate(labelled.iterrows()):
    _, make, type_, model, _ = cascade(osd[i][0], r["mac_is_random"], *old[i])
    truth = (r["make"], r["type"], r["model"])
    if tuple(old[i]) == truth and (make, type_, model) != truth:
        broken += 1
print(f"capture có nhãn: {len(labelled)} cửa sổ, cascade làm hỏng {broken} câu đúng")

iot = prep(sessions[~is_field])
old = iden.run(["output"], {"input": iden_matrix(iot)})[0]
osd = os_decide(os_bundle, iot)
leaked = sum(cascade(osd[i][0], iot["mac_is_random"].iat[i], *old[i])[2] in GENERAL_PURPOSE_TYPES
             for i in range(len(iot)))
print(f"CIC-2022 (in-sample): {leaked}/{len(iot)} cửa sổ IoT bị cascade gán type đa dụng")

capture có nhãn: 50 cửa sổ, cascade làm hỏng 0 câu đúng

CIC-2022 (in-sample): 0/8139 cửa sổ IoT bị cascade gán type đa dụng